# M2 Notebook 30 — Integrated Machine-Learning Case Studies

**Status:** Runnable first edition

## Learning objectives

- Integrate prediction, calibration, costs, fairness, drift, and monitoring.
- Build end-to-end Decision Intelligence workflows.

In [ ]:
from srai_math.utils import environment_info,set_seed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()
from srai_ml import (
    LogisticRegressionGD,StandardScaler,accuracy_score,brier_score,
    cost_sensitive_decision,drift_population_stability_index,
    fairness_group_rates,optimal_threshold,rank_by_expected_value,
    train_test_split,
)


## Case 1 — Risk model with calibrated decision threshold

In [ ]:
rng=np.random.default_rng(30)
n=2500
X=rng.normal(size=(n,4))
group=np.where(X[:,3]>0,"Group_A","Group_B")
logit=-1+1.4*X[:,0]-.8*X[:,1]+.3*X[:,2]
prob_true=1/(1+np.exp(-logit))
y=rng.binomial(1,prob_true)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.3,seed=30)
scaler=StandardScaler()
Xtr_s=scaler.fit_transform(Xtr); Xte_s=scaler.transform(Xte)
model=LogisticRegressionGD(.1,3000,l2=.001).fit(Xtr_s,ytr)
prob=model.predict_proba(Xte_s)[:,1]
threshold,cost=optimal_threshold(yte,prob,false_positive_cost=1,false_negative_cost=4)
pred=cost_sensitive_decision(prob,threshold)
{"threshold":threshold,"accuracy":accuracy_score(yte,pred),
 "brier":brier_score(yte,prob),"decision_cost":cost}


## Case 2 — Fairness diagnostics

In [ ]:
test_groups=group[np.setdiff1d(np.arange(n),np.arange(len(Xtr)))]
# Reconstruct consistent groups using the same split indices for demonstration
_,Xte2,_,gte=train_test_split(X,group,test_size=.3,seed=30)
rates=fairness_group_rates(yte,pred,gte)
rates


## Case 3 — Expected-value prioritization

In [ ]:
value=rank_by_expected_value(prob,benefit=10,cost=2)
top=np.argsort(value)[::-1][:20]
{"mean_probability_top20":prob[top].mean(),
 "mean_expected_value_top20":value[top].mean()}


## Case 4 — Drift monitoring

In [ ]:
reference=Xtr[:,0]
current=Xte[:,0]+.35
psi=drift_population_stability_index(reference,current,10)
{"population_stability_index":psi}


## Governance checklist

- Validate data lineage.
- Preserve test integrity.
- Calibrate probabilities.
- Select thresholds from explicit costs.
- Assess subgroup performance.
- Monitor drift.
- Maintain human review.

## Key insight

A production ML system is a governed decision process, not merely a trained model.